### . Import libraries

In [12]:
from pathlib import Path
import json, numpy as np, pandas as pd, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

### load datasets

In [13]:
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
DATA = ROOT / "datasets/processed"

train_df = pd.read_csv(
    DATA / "paddy_train.csv",
    keep_default_na=False,
    encoding="utf-8-sig"
)

val_df = pd.read_csv(
    DATA / "paddy_validation.csv",
    keep_default_na=False,
    encoding="utf-8-sig"
)

### Convert category labels into numbers

In [14]:
labels = sorted(train_df["label"].unique().tolist())

label2id = {v:i for i,v in enumerate(labels)}

id2label = {i:v for v,i in label2id.items()}

assert set(val_df["label"]).issubset(set(labels)), \
    "Validation contains a category missing from training."

### Convert CSV data into Hugging Face Dataset

In [15]:
MODEL_NAME = "FacebookAI/xlm-roberta-base"
MAX_LENGTH = 160
def to_dataset(frame):
    data = frame[["question", "label"]].copy()
    data["label"] = data["label"].map(label2id).astype(int)
    return Dataset.from_pandas(data, preserve_index=False)

train_ds, val_ds = to_dataset(train_df), to_dataset(val_df)

### Tokeniz


In [16]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(batch):
    return tokenizer(
        batch["question"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/17760 [00:00<?, ? examples/s]

Map:   0%|          | 0/3840 [00:00<?, ? examples/s]

###  Load the classification model

In [17]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    label2id=label2id,
    id2label=id2label
)

def compute_metrics(eval_pred):
    logits, gold = eval_pred
    pred = np.argmax(logits, axis=1)

    p, r, f1, _ = precision_recall_fscore_support(
        gold,
        pred,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy_score(gold, pred),
        "precision_macro": p,
        "recall_macro": r,
        "f1_macro": f1
    }

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### training


In [18]:
out_dir = ROOT / "models/problem_classifier/xlm_roberta_base"

args = TrainingArguments(
    output_dir=str(out_dir),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    report_to="none",
    seed=42,
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,0.000749,3.991475,0.520833,0.522768,0.494097,0.484110
2,0.003607,4.344877,0.508333,0.517418,0.464236,0.454660
3,0.000139,4.569044,0.503125,0.437544,0.458333,0.432637


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6660, training_loss=0.22773537686368725, metrics={'train_runtime': 1676.4459, 'train_samples_per_second': 31.782, 'train_steps_per_second': 3.973, 'total_flos': 1239119705256480.0, 'train_loss': 0.22773537686368725, 'epoch': 3.0})

### Evaluate the trained model

In [19]:
val_output = trainer.predict(val_ds)

val_pred_ids = np.argmax(val_output.predictions, axis=1)

val_pred = [id2label[int(i)] for i in val_pred_ids]

### Calculate final validation metrics

In [20]:
metrics = compute_metrics(
    (val_output.predictions, val_output.label_ids)
)

metrics = {k: float(v) for k, v in metrics.items()}

print("Validation metrics:", json.dumps(metrics, indent=2))

Validation metrics: {
  "accuracy": 0.5208333333333334,
  "precision_macro": 0.5227682609894096,
  "recall_macro": 0.4940972222222222,
  "f1_macro": 0.48410983591280804
}


### Save the best model

In [21]:
trainer.save_model(str(out_dir / "best_model"))

tokenizer.save_pretrained(str(out_dir / "best_model"))

(ROOT / "models/problem_classifier/label_map_xlm_roberta.json").write_text(
    json.dumps(
        {
            "label2id": label2id,
            "id2label": {str(k): v for k, v in id2label.items()}
        },
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

pd.DataFrame({
    "question": val_df["question"],
    "language": val_df["language"],
    "issue_id": val_df["issue_id"],
    "true_label": val_df["label"],
    "predicted_label": val_pred
}).to_csv(
    ROOT / "models/model_evaluation_results/xlmr_validation_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]